# Day 1: DNA counts, logistic regression and random forest

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/palexiou/ML_Course_2026/blob/main/notebooks/01_simple_signal.ipynb)

Completed workflow generated from `course_code/`; edit the canonical Python modules and rebuild with `python tools/build_course.py`.

Run setup and both definition cells, then follow the selected Section 3 cells below. Each notebook contains its Python dependencies; opening it does not clone the repository. This is controlled simulated DNA classification, not real promoter discovery. Students may follow along at selected cells or watch the instructor demonstration.

See [execution and access instructions](https://github.com/palexiou/ML_Course_2026/blob/main/notebooks/README.md) and the separate scaffolded practicals and instructor answers. Save a Drive copy to retain edits.

The primary dataset samples a fresh 15-base site from a fixed positive-entry PPM at each planting event (0.95 for positives, 0.05 for negatives), at a uniform valid forward-strand start. Defaults: 1,200 balanced 160-base sequences, seed 2026, 720/240/240 splits. Old fixed-string datasets/caches need regeneration; see the notebook guide. The fixed-position interaction and family-memorization controls retain separate rules.

**Classroom route:** GitHub notebook → Open in Colab → setup → definition cells → selected Section 3 cells below. Stop before the later full workflow; Run all would also fit the forest and print validation metrics. The short path is complete working code for an instructor demo or optional student follow-along.


In [ ]:
# Install missing direct dependencies only; existing packages are not silently replaced.
# Exact locally rehearsed versions below are a baseline, not verified Colab compatibility.
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys
requirements = ['numpy==2.3.2', 'pandas==2.3.1', 'scikit-learn==1.7.2', 'matplotlib==3.10.5']
missing = []
for requirement in requirements:
    package, expected = requirement.split('==')
    try:
        installed = version(package)
        print(f'{package}: {installed} (local baseline {expected})')
    except PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print('If prompted, restart the runtime, then rerun this notebook from the top.')


In [ ]:
"""Shared simulation, evaluation, and artifact contracts for the course."""

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

BASE_ORDER = "ACGT"
PRIMARY_GENERATOR_VERSION = "independent-ppm15-forward-v2"
DATASET_SCHEMA_VERSION = 2
# Rows are positions, columns A/C/G/T. Fixed independently of the dataset seed.
DEFAULT_MOTIF_PPM = (
    (0.05, 0.05, 0.10, 0.80),
    (0.70, 0.10, 0.10, 0.10),
    (0.05, 0.05, 0.15, 0.75),
    (0.65, 0.15, 0.10, 0.10),
    (0.50, 0.40, 0.05, 0.05),
    (0.60, 0.10, 0.20, 0.10),
    (0.10, 0.10, 0.10, 0.70),
    (0.10, 0.10, 0.70, 0.10),
    (0.10, 0.65, 0.15, 0.10),
    (0.15, 0.10, 0.65, 0.10),
    (0.55, 0.15, 0.20, 0.10),
    (0.10, 0.60, 0.10, 0.20),
    (0.10, 0.10, 0.75, 0.05),
    (0.10, 0.10, 0.15, 0.65),
    (0.15, 0.55, 0.20, 0.10),
)

@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 2026
    sample_count: int = 1200
    sequence_length: int = 160
    # Array inputs are copied into immutable tuples during validation.
    motif_ppm: tuple[tuple[float, ...], ...] | np.ndarray = DEFAULT_MOTIF_PPM
    positive_plant_probability: float = 0.95
    negative_plant_probability: float = 0.05
    kmer_length: int = 4

    def __post_init__(self) -> None:
        if not 0 <= self.seed < 2**32:
            raise ValueError("seed must be between 0 and 2**32 - 1")
        if self.sample_count < 20 or self.sample_count % 2:
            raise ValueError("sample_count must be an even integer of at least 20")
        matrix = np.array(self.motif_ppm, dtype=float, copy=True)
        if matrix.shape != (15, 4):
            raise ValueError("motif_ppm must have shape (15 positions, 4 ACGT bases)")
        if not np.isfinite(matrix).all() or np.any(matrix <= 0) or np.any(matrix >= 1):
            raise ValueError("motif_ppm entries must be finite, strictly positive and less than one")
        if not np.allclose(matrix.sum(axis=1), 1.0, rtol=0, atol=1e-12):
            raise ValueError("motif_ppm rows must sum to one")
        if np.any(matrix.max(axis=1) < 0.5):
            raise ValueError("Each motif_ppm position needs a leading probability of at least 0.5")
        object.__setattr__(self, "motif_ppm", tuple(tuple(float(p) for p in row) for row in matrix))
        if self.sequence_length < len(self.motif_ppm):
            raise ValueError("sequence_length must be at least the motif length")
        if not 1 <= self.kmer_length <= self.sequence_length:
            raise ValueError("kmer_length must be between 1 and sequence_length")
        for probability in (
            self.positive_plant_probability,
            self.negative_plant_probability,
        ):
            if not 0 <= probability <= 1:
                raise ValueError("motif planting probabilities must be between 0 and 1")


def simulation_settings(config: SimulationConfig) -> dict[str, Any]:
    return {"dataset_variant": "simple_signal", "algorithm_version": PRIMARY_GENERATOR_VERSION,
            "base_order": BASE_ORDER, "background_probabilities": [0.25] * 4,
            "orientation": "forward_only", **asdict(config)}


def settings_fingerprint(settings: dict[str, Any]) -> str:
    return hashlib.sha256(json.dumps(settings, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def generation_contract() -> dict[str, Any]:
    return {"schema_version": DATASET_SCHEMA_VERSION,
            "primary_defaults": simulation_settings(SimulationConfig())}


def expected_motif_kmers(config: SimulationConfig, words: Any) -> pd.DataFrame:
    """Average internal-window probability; enrichment denominator is uniform 4^-k.

    Excludes windows crossing site boundaries, label mixtures and full-sequence counts.
    For k > 15 there are no wholly internal windows, so the annotation is undefined.
    """
    k = config.kmer_length
    if k > 15:
        return pd.DataFrame(columns=["kmer", "motif_window_probability", "motif_vs_background_enrichment"])
    matrix = np.asarray(config.motif_ppm)
    records = []
    for word in words:
        if len(word) != k or set(word) - set(BASE_ORDER):
            raise ValueError("Expected ACGT words of the configured k-mer length")
        bases = [BASE_ORDER.index(base) for base in word]
        probability = sum(float(np.prod(matrix[start + np.arange(k), bases]))
                          for start in range(16 - k)) / (16 - k)
        records.append({"kmer": word,
                        "motif_window_probability": probability,
                        "motif_vs_background_enrichment": probability / (0.25 ** k)})
    return pd.DataFrame(records)


def sample_motif_site(random: np.random.Generator, config: SimulationConfig) -> str:
    # Independent draws for every position of every inserted site.
    return "".join(random.choice(list(BASE_ORDER), p=row) for row in config.motif_ppm)


def generate_course_dataset(
    config: SimulationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    random = np.random.default_rng(config.seed)
    labels = np.repeat([0, 1], config.sample_count // 2)
    random.shuffle(labels)
    sequences = random.choice(list("ACGT"), size=(config.sample_count, config.sequence_length))
    planting_probabilities = np.where(
        labels == 1,
        config.positive_plant_probability,
        config.negative_plant_probability,
    )
    planted = random.random(config.sample_count) < planting_probabilities
    starts = np.full(config.sample_count, -1, dtype=int)
    sites: list[str | None] = [None] * config.sample_count
    for index in np.flatnonzero(planted):
        start = int(random.integers(0, config.sequence_length - len(config.motif_ppm) + 1))
        site = sample_motif_site(random, config)
        sequences[index, start : start + len(config.motif_ppm)] = list(site)
        starts[index] = start
        sites[index] = site
    sample_ids = [f"simple-{config.seed}-{index:05d}" for index in range(config.sample_count)]
    samples = pd.DataFrame(
        {
            "sample_id": sample_ids,
            "group_id": sample_ids,
            "sequence": ["".join(sequence) for sequence in sequences],
            "label": labels,
        }
    )
    if samples["sequence"].duplicated().any():
        raise ValueError("The independent-sample variant contains duplicate sequences")
    truth = pd.DataFrame(
        {"sample_id": sample_ids, "motif_planted": planted,
         "motif_start": pd.array([int(start) if flag else None for start, flag in zip(starts, planted)], dtype="Int64"),
         "sampled_site": sites}
    )
    return samples, truth


def assign_splits(samples: pd.DataFrame, seed: int) -> pd.DataFrame:
    train_indices, heldout_indices = train_test_split(
        samples.index,
        test_size=0.4,
        stratify=samples["label"],
        random_state=seed,
    )
    validation_indices, test_indices = train_test_split(
        heldout_indices,
        test_size=0.5,
        stratify=samples.loc[heldout_indices, "label"],
        random_state=seed,
    )
    result = samples.copy()
    result["split"] = "train"
    result.loc[validation_indices, "split"] = "validation"
    result.loc[test_indices, "split"] = "test"
    if result.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if len(train_indices) + len(validation_indices) + len(test_indices) != len(result):
        raise ValueError("Splits do not cover the course dataset")
    return result



def classification_metrics(labels: Any, probabilities: Any, evaluation_split: str) -> dict[str, Any]:
    if evaluation_split not in {"validation", "test"}:
        raise ValueError("Evaluation split must be validation or test")
    if not np.isfinite(probabilities).all() or np.any(np.asarray(probabilities) < 0) or np.any(np.asarray(probabilities) > 1):
        raise ValueError("Probabilities must be finite and between zero and one")
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    return {
        "evaluation_split": evaluation_split, "primary_metric": "roc_auc", "decision_threshold": 0.5,
        "sample_count": len(labels), "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }


def validation_metrics(labels: Any, probabilities: Any) -> dict[str, Any]:
    return classification_metrics(labels, probabilities, "validation")


def write_json(path: str | Path, value: Any) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_hash(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def sample_hash(samples: pd.DataFrame) -> str:
    return hashlib.sha256(samples.to_csv(index=False, lineterminator="\n").encode()).hexdigest()


def save_dataset(root: str | Path, samples: pd.DataFrame, truth: pd.DataFrame, settings: dict[str, Any]) -> None:
    if "motif" in settings or ("motif_ppm" in settings and settings.get("algorithm_version") != PRIMARY_GENERATOR_VERSION):
        raise ValueError("Legacy primary settings: regenerate using simulation_settings(config)")
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        samples.loc[samples["split"] == split].to_csv(root / filename, index=False)
    samples[["sample_id", "group_id", "split"]].to_csv(root / "split_manifest.csv", index=False)
    truth.to_csv(root / "simulation_truth.csv", index=False)
    write_json(root / "generation.json", {
        "schema_version": DATASET_SCHEMA_VERSION, "settings": settings,
        "settings_sha256": settings_fingerprint(settings), "dataset_sha256": sample_hash(samples),
        "files": {name: file_hash(root / name) for name in
                  ("train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv")},
    })


def validate_generation(root: str | Path) -> dict[str, Any]:
    """Reject legacy generation and tampered data before development/freeze reads."""
    root = Path(root)
    metadata = json.loads((root / "generation.json").read_text(encoding="utf-8"))
    if metadata.get("schema_version") != DATASET_SCHEMA_VERSION:
        raise ValueError("Incompatible dataset metadata; regenerate development artifacts in a new directory")
    settings = metadata["settings"]
    if "motif" in settings:
        raise ValueError("Legacy fixed-motif dataset; regenerate with the PPM generator")
    if settings.get("dataset_variant") == "simple_signal" or "motif_ppm" in settings:
        fields = SimulationConfig.__dataclass_fields__
        config = SimulationConfig(**{key: settings[key] for key in fields})
        if settings_fingerprint(settings) != settings_fingerprint(simulation_settings(config)):
            raise ValueError("Incompatible primary generator algorithm or settings")
    if metadata.get("settings_sha256") != settings_fingerprint(settings):
        raise ValueError("Generation settings fingerprint mismatch")
    names = {"train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv"}
    if set(metadata.get("files", {})) != names:
        raise ValueError("Incomplete dataset file identity metadata")
    for name in names:
        if file_hash(root / name) != metadata["files"][name]:
            raise ValueError(f"Dataset file identity mismatch: {name}")
    return metadata


def read_dataset(root: str | Path, include_test_labels: bool = False) -> pd.DataFrame:
    root = Path(root)
    validate_generation(root)
    partitions = []
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        if split == "test" and not include_test_labels:
            partition = pd.read_csv(root / filename, usecols=lambda column: column != "label")
            partition["label"] = np.nan
        else:
            partition = pd.read_csv(root / filename)
        required = {"sample_id", "group_id", "sequence", "label", "split"}
        non_null = required - ({"label"} if split == "test" and not include_test_labels else set())
        if not required <= set(partition) or partition[list(non_null)].isna().any().any():
            raise ValueError("Dataset columns are missing or contain null values")
        labels_expected = split != "test" or include_test_labels
        if partition.empty or not partition["split"].eq(split).all() or (labels_expected and set(partition["label"]) != {0, 1}):
            raise ValueError("A partition has an incorrect split or lacks both binary classes")
        partitions.append(partition)
    samples = pd.concat(partitions, ignore_index=True)
    if not samples["sample_id"].is_unique or not samples["sequence"].map(lambda sequence: bool(sequence) and set(sequence) <= set("ACGT")).all():
        raise ValueError("Dataset IDs or DNA sequences are invalid")
    if samples.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if samples.groupby("sequence")["split"].nunique().max() != 1:
        raise ValueError("Duplicate DNA sequences cross split boundaries")
    manifest = pd.read_csv(root / "split_manifest.csv")
    expected = samples[["sample_id", "group_id", "split"]].sort_values("sample_id").reset_index(drop=True)
    actual = manifest.sort_values("sample_id").reset_index(drop=True)
    if not expected.equals(actual):
        raise ValueError("Split manifest does not match sample identities")
    samples.attrs["dataset_generation_sha256"] = file_hash(root / "generation.json")
    return samples


def build_interaction_dataset(seed: int = 2026, pair_count: int = 240) -> tuple[pd.DataFrame, pd.DataFrame]:
    if pair_count < 20:
        raise ValueError("At least 20 independent pairs are required")
    random = np.random.default_rng(seed)
    motif_first, motif_second = "TATAAATG", "CGGACGTC"
    flank_left, flank_right = "GGGGGG", "CCCCCC"
    positions = (32, 80)
    records, truth = [], []
    for pair_index in range(pair_count):
        background = random.choice(list("ACGT"), size=160)
        group = f"order-{seed}-{pair_index:05d}"
        for label, motifs in ((1, (motif_first, motif_second)), (0, (motif_second, motif_first))):
            sequence = background.copy()
            for position, motif in zip(positions, motifs):
                block = flank_left + motif + flank_right
                sequence[position:position + len(block)] = list(block)
            sample_id = f"{group}-{label}"
            records.append({"sample_id": sample_id, "group_id": group, "sequence": "".join(sequence), "label": label})
            truth.append({"sample_id": sample_id, "first_block_start": positions[0], "second_block_start": positions[1], "first_motif": motifs[0], "second_motif": motifs[1]})
    samples = pd.DataFrame(records)
    groups = samples["group_id"].unique()
    train_groups, heldout_groups = train_test_split(groups, test_size=0.4, random_state=seed)
    validation_groups, test_groups = train_test_split(heldout_groups, test_size=0.5, random_state=seed)
    samples["split"] = "train"
    samples.loc[samples["group_id"].isin(validation_groups), "split"] = "validation"
    samples.loc[samples["group_id"].isin(test_groups), "split"] = "test"
    return samples, pd.DataFrame(truth)


## Stage 01: Day 1: DNA counts, logistic regression and random forest

Run the definition cell, then follow Section 3 below; leave the complete workflow for later. Definitions do not train models. Generated outputs are temporary runtime files.


In [ ]:
"""A standalone simulated DNA classification experiment for a script or Colab cell."""

from importlib.metadata import version
from pathlib import Path
from time import perf_counter
from typing import Any
import hashlib
import json
import platform

IMPORTS_STARTED = perf_counter()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

IMPORT_SECONDS = perf_counter() - IMPORTS_STARTED
PRIMARY_METRIC = "roc_auc"
DECISION_THRESHOLD = 0.5




def run_experiment(
    output_dir: str | Path = "simple_signal_output",
    config: SimulationConfig | None = None,
    show_plot: bool = True,
) -> dict[str, Any]:
    started = perf_counter()
    config = config if config is not None else SimulationConfig()
    destination = Path(output_dir)
    if any((destination / marker).exists() for marker in ("frozen_manifest.json", "test_results.json")):
        raise RuntimeError("This experiment is frozen or revealed. Use a new development directory.")
    destination.mkdir(parents=True, exist_ok=True)
    samples, truth = generate_course_dataset(config)
    samples = assign_splits(samples, config.seed)
    train = samples.loc[samples["split"] == "train"]
    validation = samples.loc[samples["split"] == "validation"]
    generation_seconds = perf_counter() - started

    pipeline = Pipeline(
        [
            (
                "kmers",
                CountVectorizer(
                    analyzer="char",
                    ngram_range=(config.kmer_length, config.kmer_length),
                    lowercase=False,
                ),
            ),
            ("classifier", LogisticRegression(max_iter=2000)),
        ]
    )
    fit_started = perf_counter()
    pipeline.fit(train["sequence"], train["label"])
    fit_seconds = perf_counter() - fit_started
    evaluation_started = perf_counter()
    probabilities = pipeline.predict_proba(validation["sequence"])[:, 1]
    predictions = (probabilities >= DECISION_THRESHOLD).astype(int)
    metrics = {
        "primary_metric": PRIMARY_METRIC,
        "primary_metric_reason": "Ranking discrimination on balanced simulated labels; fixed before fitting.",
        "evaluation_split": "validation",
        "decision_threshold": DECISION_THRESHOLD,
        "sample_count": len(validation),
        "positive_prevalence": float(validation["label"].mean()),
        "roc_auc": float(roc_auc_score(validation["label"], probabilities)),
        "average_precision": float(average_precision_score(validation["label"], probabilities)),
        "accuracy": float(accuracy_score(validation["label"], predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(validation["label"], predictions)),
        "precision": float(precision_score(validation["label"], predictions, zero_division=0)),
        "recall": float(recall_score(validation["label"], predictions, zero_division=0)),
        "confusion_matrix": confusion_matrix(validation["label"], predictions, labels=[0, 1]).tolist(),
    }
    evaluation_seconds = perf_counter() - evaluation_started

    save_dataset(destination, samples, truth, simulation_settings(config))
    validation_predictions = validation[["sample_id", "label"]].copy()
    validation_predictions["probability"] = probabilities
    validation_predictions["prediction"] = predictions
    validation_predictions.to_csv(destination / "validation_predictions.csv", index=False)
    (destination / "validation_metrics.json").write_text(
        json.dumps(metrics, indent=2) + "\n", encoding="utf-8"
    )
    vectorizer = pipeline.named_steps["kmers"]
    classifier = pipeline.named_steps["classifier"]
    annotations = expected_motif_kmers(config, vectorizer.get_feature_names_out())
    pd.DataFrame(
        {"kmer": vectorizer.get_feature_names_out(), "coefficient": classifier.coef_[0]}
    ).merge(annotations, on="kmer", how="left").sort_values("coefficient", ascending=False).to_csv(
        destination / "kmer_coefficients.csv", index=False
    )

    forest = RandomForestClassifier(n_estimators=100, random_state=config.seed, n_jobs=2)
    forest_started = perf_counter()
    forest.fit(vectorizer.transform(train["sequence"]), train["label"])
    forest_fit_seconds = perf_counter() - forest_started
    forest_probabilities = forest.predict_proba(vectorizer.transform(validation["sequence"]))[:, 1]
    forest_metrics = validation_metrics(validation["label"], forest_probabilities)
    write_json(destination / "random_forest_validation_metrics.json", forest_metrics)
    forest_predictions = validation[["sample_id", "group_id", "label"]].copy()
    forest_predictions["probability"] = forest_probabilities
    forest_predictions.to_csv(destination / "random_forest_validation_predictions.csv", index=False)
    importance = pd.DataFrame({"kmer": vectorizer.get_feature_names_out(), "importance": forest.feature_importances_})
    importance = importance.merge(annotations, on="kmer", how="left")
    importance.sort_values("importance", ascending=False).to_csv(destination / "random_forest_importance.csv", index=False)
    comparison = pd.DataFrame([{"method": "logistic_regression", **metrics}, {"method": "random_forest", **forest_metrics}])
    comparison.to_csv(destination / "classical_validation.csv", index=False)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    ConfusionMatrixDisplay.from_predictions(
        validation["label"], predictions, labels=[0, 1], ax=axes[0], colorbar=False
    )
    axes[0].set_title("Validation confusion matrix (threshold 0.5)")
    RocCurveDisplay.from_predictions(
        validation["label"], probabilities, ax=axes[1], plot_chance_level=True
    )
    axes[1].set_title("Validation ROC curve")
    PrecisionRecallDisplay.from_predictions(
        validation["label"], probabilities, ax=axes[2], plot_chance_level=True
    )
    axes[2].set_title("Validation precision-recall curve (AP)")
    figure.tight_layout()
    figure.savefig(destination / "validation_curves.png", dpi=150)

    metadata = {
        "dataset_variant": "simple_signal",
        "simulation": simulation_settings(config),
        "settings_sha256": settings_fingerprint(simulation_settings(config)),
        "simulation_limitations": "Independent uniform-background sequences, not real promoters. Labels define a planted-motif mixture; motif presence is not a deterministic label rule.",
        "group_policy": "Each independently generated sequence is a singleton group; this does not infer biological homology.",
        "truth_policy": "motif_planted records deliberate insertion, not a scan or consensus match. Unplanted start/site are null. Truth fields are never model inputs.",
        "kmer_annotation": "motif_window_probability is the mean product of PPM probabilities across the 15-k+1 wholly internal windows. Enrichment divides by uniform-background probability 4^-k. Boundary windows and label mixtures are excluded; undefined for k>15. All words have positive probability, so there is no binary motif membership.",
        "split_counts": {name: int(count) for name, count in samples["split"].value_counts().items()},
        "split_policy": "Stratified 60/20/20 with recorded sample identities; reuse the manifest across representations.",
        "feature_fit_split": "train",
        "feature_count": len(vectorizer.get_feature_names_out()),
        "classifier_parameters": classifier.get_params(),
        "random_forest_parameters": forest.get_params(),
        "importance_caution": "Impurity-based importance is not causal evidence or proof of a biological motif; correlated overlapping k-mers share signal.",
        "protected_test_evaluated": False,
        "test_policy": "Exported for the planned Day 3 comparison only. File separation is a teaching convention, not an access-control boundary.",
        "dataset_sha256": hashlib.sha256(
            samples.to_csv(index=False, lineterminator="\n").encode("utf-8")
        ).hexdigest(),
        "runtime": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "packages": {
                name: version(name)
                for name in ("numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "joblib", "threadpoolctl")
            },
        },
        "timing_seconds": {
            "imports": IMPORT_SECONDS,
            "generation_and_split": generation_seconds,
            "fit": fit_seconds,
            "random_forest_fit": forest_fit_seconds,
            "validation_evaluation": evaluation_seconds,
            "total": perf_counter() - started + IMPORT_SECONDS,
        },
    }
    (destination / "experiment_metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
    )
    print("SIMULATION: simplified DNA classification, not real promoter recognition.")
    print("Primary metric chosen before fitting: validation ROC-AUC.")
    print(json.dumps(metrics, indent=2))
    print(comparison[["method", "roc_auc", "average_precision"]].to_string(index=False))
    print("Protected test sequences exported but NOT evaluated. Do not tune against them.")
    print(f"Artifacts: {destination.resolve()}")
    if show_plot:
        plt.show()
    plt.close(figure)
    return metrics


## Section 3: from DNA counts to probability and loss

Open this completed GitHub notebook with **Open in Colab**, run **setup** and
the two **definition** cells above, then run the five steps below in order.
An instructor can select these cells for a roughly five-minute demonstration
after setup; students may follow along or try the small changes suggested here.
No coding from scratch or submission is required. Setup time is separate;
this is a suggested route, not a measured classroom timing.

We use **TRAIN / VALIDATION ONLY**. Predict one validation record before
revealing its known label, then ask how much loss that prediction incurs.
Stop at the marked boundary before validation metrics and forests.


### 1. Prepare the same primary dataset

Use the canonical 15-position probability motif, 0.95/0.05 planting and
160-base sequences. The generator saves the complete fixed split in the normal
`simple_signal_output/` location. Only training and validation tables are
loaded for this lesson; test file bytes are hashed solely to check identity.
Existing artifacts must match the canonical configuration and file hashes.


In [ ]:
s3_config = SimulationConfig()
s3_destination = Path("simple_signal_output")
if any((s3_destination / marker).exists()
       for marker in ("frozen_manifest.json", "test_results.json")):
    raise RuntimeError("This experiment is frozen or revealed. Use a new development directory.")
if not (s3_destination / "generation.json").exists():
    if s3_destination.exists() and any(s3_destination.iterdir()):
        raise ValueError("Existing artifacts lack generation identity; preserve them and use a fresh runtime.")
    s3_generated, s3_truth = generate_course_dataset(s3_config)
    s3_generated = assign_splits(s3_generated, s3_config.seed)
    save_dataset(s3_destination, s3_generated, s3_truth, simulation_settings(s3_config))
    del s3_generated, s3_truth
s3_generation = validate_generation(s3_destination)
assert s3_generation["settings_sha256"] == settings_fingerprint(simulation_settings(s3_config))
s3_train = pd.read_csv(s3_destination / "train.csv")
s3_validation = pd.read_csv(s3_destination / "validation.csv")
assert s3_train["split"].eq("train").all()
assert s3_validation["split"].eq("validation").all()
assert set(s3_train.sample_id).isdisjoint(s3_validation.sample_id)
assert set(s3_train.group_id).isdisjoint(s3_validation.group_id)
print(f"TRAIN: {len(s3_train)}; VALIDATION: {len(s3_validation)}. No test table loaded.")
print("Settings SHA-256:", s3_generation["settings_sha256"])
print("Generation SHA-256:", file_hash(s3_destination / "generation.json"))


### 2. Inspect DNA and its count row

Fit the vocabulary on **training sequences only**; transform validation with
that same vocabulary. Only DNA enters the vectorizer: labels, IDs, groups,
split and planting truth are excluded. The model uses every count, including
zeros; we display ten nonzero entries to keep the row readable.

Optional: change `s3_row_index` to `1`, then rerun this step and steps 3–5.
Which counts and probability change? Keep model and dataset settings fixed.


In [ ]:
s3_vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(s3_config.kmer_length, s3_config.kmer_length),
    lowercase=False,
)
s3_x_train = s3_vectorizer.fit_transform(s3_train["sequence"])
s3_x_validation = s3_vectorizer.transform(s3_validation["sequence"])
s3_words = s3_vectorizer.get_feature_names_out()
s3_row_index = 0  # Optional: try another validation row; rerun steps 2–5.
s3_record = s3_validation.iloc[s3_row_index]
s3_x_row = s3_x_validation[s3_row_index]
s3_count_row = pd.Series(s3_x_row.toarray().ravel(), index=s3_words, name="count")
print("Validation ID (metadata only):", s3_record["sample_id"])
print("DNA:", s3_record["sequence"])
print(f"{len(s3_words)} training-vocabulary features; count sum = {s3_count_row.sum()}")
print(s3_count_row[s3_count_row > 0].head(10).to_string())


### 3. Learn the weights from training labels

Each k-mer gets a coefficient; the intercept is the bias. A positive
coefficient raises the score for class B when that count increases and
other counts stay fixed. These learned associations are not causal evidence.
We keep the same logistic-regression defaults as the complete workflow.


In [ ]:
s3_classifier = LogisticRegression(max_iter=2000)
s3_classifier.fit(s3_x_train, s3_train["label"])
assert s3_classifier.classes_.tolist() == [0, 1]
s3_coefficients = pd.Series(s3_classifier.coef_[0], index=s3_words, name="weight")
print("Class mapping: A = 0, B = 1; predict_proba columns:", s3_classifier.classes_)
print("Intercept (bias):", s3_classifier.intercept_[0])
print("First ten coefficients in vocabulary order:")
print(s3_coefficients.head(10).to_string())


### 4. Reproduce one prediction by hand

The score is $z = x \cdot w + b$. The sigmoid turns it into
$p = P(B=1) = 1/(1+e^{-z})$; $P(A=0) = 1-p$.
`decision_function` returns $z$; `predict_proba` returns both probabilities
in `classes_` order. `expit` is a numerically stable sigmoid.

Optional: inspect `s3_count_row * s3_coefficients` to see the individual
contributions. Can you explain why a zero count contributes zero?


In [ ]:
from scipy.special import expit

s3_probabilities = s3_classifier.predict_proba(s3_x_row)[0]
s3_score = float(s3_classifier.decision_function(s3_x_row)[0])
s3_manual_z = float(s3_count_row.to_numpy() @ s3_classifier.coef_[0]
                    + s3_classifier.intercept_[0])
s3_manual_p = float(expit(s3_manual_z))
np.testing.assert_allclose(s3_manual_z, s3_score, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose([1 - s3_manual_p, s3_manual_p], s3_probabilities,
                           rtol=1e-10, atol=1e-12)
print(f"decision_function z = {s3_score:.6f}; manual x·w+b = {s3_manual_z:.6f}")
print(f"predict_proba: P(A=0) = {s3_probabilities[0]:.6f}; P(B=1) = {s3_probabilities[1]:.6f}")
print(f"Manual sigmoid(z) = {s3_manual_p:.6f}. Known label is revealed next.")


### 5. Compare the prediction with the known label: loss

For $y \in \{0,1\}$, binary cross-entropy is
$L = -[y\log(p) + (1-y)\log(1-p)]$ (natural logarithms).
If $y=1$, this is $-\log(p)$; if $y=0$, it is $-\log(1-p)$.
A confident correct prediction has small loss; a confident wrong prediction
has large loss. `logaddexp(0, z) - y*z` evaluates the same loss stably.

Training averages this loss across records. Below we inspect its **data-loss
part**; scikit-learn also applies its default L2 regularization penalty to
the weights. Mean training loss is not a claim of generalization.
We leave validation performance summaries for Section 4.


In [ ]:
from sklearn.metrics import log_loss

s3_y = int(s3_record["label"])  # Reveal the observed label after predicting.
s3_row_loss = float(np.logaddexp(0.0, s3_manual_z) - s3_y * s3_manual_z)
s3_sklearn_row_loss = log_loss([s3_y], [s3_probabilities], labels=s3_classifier.classes_)
np.testing.assert_allclose(s3_row_loss, s3_sklearn_row_loss, rtol=1e-10, atol=1e-12)
s3_train_z = s3_classifier.decision_function(s3_x_train)
s3_mean_training_loss = float(np.mean(
    np.logaddexp(0.0, s3_train_z) - s3_train["label"].to_numpy() * s3_train_z
))
np.testing.assert_allclose(s3_mean_training_loss, log_loss(
    s3_train["label"], s3_classifier.predict_proba(s3_x_train), labels=s3_classifier.classes_
), rtol=1e-10, atol=1e-12)
print(f"Observed class: {'B' if s3_y else 'A'} ({s3_y}); this prediction's loss: {s3_row_loss:.6f}")
print(f"Mean TRAINING data loss: {s3_mean_training_loss:.6f} (excludes L2 penalty)")
print("Constructed examples, both with observed label B=1:")
print(pd.DataFrame({
    "prediction": ["confident correct", "confident wrong"],
    "P(B=1)": [0.95, 0.05],
    "loss = -log(p)": -np.log([0.95, 0.05]),
}).to_string(index=False))


### Optional: illustrate gradient descent on one weight

This is a small **constructed training example**, separate from the DNA fit.
Four records share one input feature $x$; both labels occur at $x=1$,
so no weight can perfectly separate them. Bias is fixed at zero.
The curve is actual mean logistic data loss as the single weight varies;
its minimum is finite and above zero.

Each update subtracts `learning_rate * gradient`: the negative gradient
points downhill locally. The orange numbered points show successive updates.
Optional: change `s3_learning_rate` from `0.8` to `0.2`, rerun, and compare
progress over eight updates. Too large a step can overshoot or increase loss.
This is not a learning-rate search on course validation data.

The DNA model has many weights plus a bias. Its scikit-learn default solver
is **L-BFGS**, not the vanilla gradient-descent loop illustrated here.


In [ ]:
s3_toy_x = np.array([1.0, 1.0, 2.0, 2.0])
s3_toy_y = np.array([0.0, 1.0, 1.0, 1.0])
s3_fixed_bias = 0.0

def s3_toy_loss(weight):
    z = weight * s3_toy_x + s3_fixed_bias
    return float(np.mean(np.logaddexp(0.0, z) - s3_toy_y * z))

def s3_toy_gradient(weight):
    z = weight * s3_toy_x + s3_fixed_bias
    return float(np.mean((expit(z) - s3_toy_y) * s3_toy_x))

s3_learning_rate = 0.8
s3_weight = -2.0
s3_steps = []
for s3_step in range(9):
    s3_gradient = s3_toy_gradient(s3_weight)
    s3_steps.append({"step": s3_step, "weight": s3_weight,
                     "loss": s3_toy_loss(s3_weight), "gradient": s3_gradient})
    if s3_step < 8:
        s3_weight -= s3_learning_rate * s3_gradient
s3_step_table = pd.DataFrame(s3_steps)
print(s3_step_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
s3_weight_grid = np.linspace(-3.0, 4.0, 250)
s3_figure, s3_ax = plt.subplots(figsize=(7, 3.8))
s3_ax.plot(s3_weight_grid, [s3_toy_loss(w) for w in s3_weight_grid],
           color="#0072B2", linewidth=2, label="Mean logistic loss (constructed data)")
s3_ax.plot(s3_step_table.weight, s3_step_table.loss, "o--", color="#D55E00",
           markersize=5, label="Negative-gradient updates")
for s3_index in (0, 1, 2, 8):
    s3_point = s3_step_table.iloc[s3_index]
    s3_ax.annotate(str(s3_index), (s3_point.weight, s3_point.loss),
                   xytext=(5, 7), textcoords="offset points", color="#D55E00")
s3_ax.set(xlabel="One weight (bias fixed at zero)", ylabel="Mean binary cross-entropy",
          title="Training means moving toward lower loss")
s3_ax.grid(axis="y", color="#dddddd", linewidth=0.6)
s3_ax.spines[["top", "right"]].set_visible(False)
s3_ax.legend(frameon=False, fontsize=9)
s3_figure.tight_layout()
plt.show()
plt.close(s3_figure)


## STOP — end of the Section 3 path

You have followed DNA → counts → learned weights → score → probability → loss.
Pause here during Section 3; the optional gradient illustration can be skipped.

## Next: Section 4 validation metrics

Continue to the focused Section 4 cells below when the instructor reaches
that lesson. They reuse this fitted model and do not train the forest.
Stop again at the Section 4 boundary. The later complete stage 01 workflow
belongs to Section 5 and preserves its normal metadata/return contract.
**Run all also runs that later paired workflow.** No protected test result
is scored. The optional gradient illustration above can be skipped.


## Section 4: inspect validation predictions

Continue from Section 3 in this same Colab runtime. Use its fitted model;
no new fitting is needed. Run the two working cells below, change only
`s4_threshold`, and explain what happens to the calls. The recorded course
comparison remains at 0.5. No test table or planting truth is loaded.


In [ ]:
# One probability per validation row, in the same row order.
s4_probabilities = s3_classifier.predict_proba(s3_x_validation)[:, 1]
s4_rows = s3_validation[["sample_id", "label"]].copy()
s4_rows["probability"] = s4_probabilities
assert s4_rows.sample_id.is_unique
s4_reference_auc = roc_auc_score(s4_rows.label, s4_probabilities)
print(s4_rows.head().to_string(index=False))
print("ROC-AUC from probabilities:", round(s4_reference_auc, 4))


In [ ]:
from sklearn.metrics import f1_score

s4_threshold = 0.5  # Then try 0.8 or 0.3; rerun this cell only.
s4_predictions = (s4_probabilities >= s4_threshold).astype(int)
s4_matrix = confusion_matrix(s4_rows.label, s4_predictions, labels=[0, 1])
s4_tn, s4_fp, s4_fn, s4_tp = s4_matrix.ravel()
s4_precision = s4_tp / (s4_tp + s4_fp) if s4_tp + s4_fp else float("nan")
s4_recall = s4_tp / (s4_tp + s4_fn) if s4_tp + s4_fn else float("nan")
print("Rows = observed A/B; columns = predicted A/B")
print(s4_matrix)
print(f"TP={s4_tp}; FP={s4_fp}; FN={s4_fn}; TN={s4_tn}")
print(f"Precision = TP/(TP+FP) = {s4_precision:.4f}")
print(f"Recall = TP/(TP+FN) = {s4_recall:.4f}")
if s4_tp + s4_fp:
    np.testing.assert_allclose(s4_precision, precision_score(s4_rows.label, s4_predictions))
else:
    print("No positive calls: precision is undefined (NaN), not observed zero.")
np.testing.assert_allclose(s4_recall, recall_score(s4_rows.label, s4_predictions))
print("F1:", round(f1_score(s4_rows.label, s4_predictions, zero_division=0), 4))
s4_auc = roc_auc_score(s4_rows.label, s4_probabilities)
assert s4_auc == s4_reference_auc  # Decisions change; scores/ranking do not.
print("Unchanged ROC-AUC:", round(s4_auc, 4))
ConfusionMatrixDisplay(s4_matrix, display_labels=["A / 0", "B / 1"]).plot(
    cmap="Blues", colorbar=False)
plt.title(f"Validation calls at threshold {s4_threshold}")
plt.show()
plt.close("all")


**STOP — end of Section 4.** Describe one changed count after moving the
threshold. This is a decision-rule exploration, not a new fitted model or
an official replacement of the 0.5 threshold. Empty positive selections
have undefined precision; libraries may use a zero-division convention.

## Section 5: compare logistic regression and random forest

At the instructor's comparison checkpoint, run the **complete stage 01
workflow immediately below once**. It regenerates the same deterministic
primary dataset and fits both models, writing their normal artifacts.
It does not reuse the live Section 3 classifier object. The forest uses
100 trees, bootstrapped training rows, and random feature candidates per split.
Validation supports comparison; test results are not scored. Read the
paired outputs in the cells after the workflow. Setup is already complete.
**Run all also runs this paired workflow.**


In [ ]:
run_experiment()


In [ ]:
s5_comparison = pd.read_csv(s3_destination / "classical_validation.csv")
assert s5_comparison.evaluation_split.eq("validation").all()
assert s5_comparison.sample_count.eq(len(s3_validation)).all()
assert s5_comparison.decision_threshold.eq(0.5).all()
print(s5_comparison[["method", "roc_auc", "average_precision", "precision", "recall"]]
      .to_string(index=False))
print("Same validation rows; same 0.5 cutoff; ROC-AUC was prespecified.")
s5_lr = pd.read_csv(s3_destination / "validation_predictions.csv")
s5_rf = pd.read_csv(s3_destination / "random_forest_validation_predictions.csv")
assert set(s5_lr.sample_id) == set(s5_rf.sample_id) == set(s3_validation.sample_id)
s5_paired = s5_lr.merge(s5_rf, on="sample_id", suffixes=("_lr", "_rf"),
                        validate="one_to_one")
assert s5_paired.label_lr.eq(s5_paired.label_rf).all()
s5_paired["prediction_rf"] = (s5_paired.probability_rf >= 0.5).astype(int)
s5_disagreements = s5_paired[s5_paired.prediction != s5_paired.prediction_rf]
s5_example = s5_disagreements.head(1) if len(s5_disagreements) else s5_paired.head(1)
print("One decision disagreement (or one probability comparison if none):")
print(s5_example[["sample_id", "label_lr", "probability_lr", "probability_rf"]]
      .to_string(index=False))
# Verify the regenerated paired workflow has the same development data.
s5_generation = validate_generation(s3_destination)
assert s5_generation == s3_generation
s5_ordered = s3_validation[["sample_id", "label"]].merge(
    s5_lr, on="sample_id", validate="one_to_one", suffixes=("_tutorial", "_run"))
assert s5_ordered.label_tutorial.eq(s5_ordered.label_run).all()
np.testing.assert_allclose(s5_ordered.probability, s4_probabilities, atol=1e-10)


**STOP — end of Section 5.** Describe what the observed comparison supports,
without assuming the forest wins or claiming a universal model ranking.
Feature-importance interpretation belongs to the next section.

## Section 6: explain a validation prediction

Reuse the Section 3 fitted classifier and count mapping for a local
explanation. Read the paired workflow's global coefficient/importance
tables afterward. No refitting, test scoring or extra data copy is needed.


In [ ]:
from scipy.special import expit

s6_row_index = 0  # Optional: select another validation row and rerun.
s6_record = s3_validation.iloc[s6_row_index]
s6_counts = s3_x_validation[s6_row_index].toarray().ravel()
s6_weights = s3_classifier.coef_[0]
s6_bias = float(s3_classifier.intercept_[0])
s6_contributions = s6_counts * s6_weights
s6_z = float(s6_bias + s6_contributions.sum())
s6_p = float(expit(s6_z))
np.testing.assert_allclose(s6_p,
    s3_classifier.predict_proba(s3_x_validation[s6_row_index])[0, 1], atol=1e-12)
s6_table = pd.DataFrame({"kmer": s3_words, "count": s6_counts,
                        "coefficient": s6_weights, "contribution": s6_contributions})
s6_top = s6_table.loc[s6_table.contribution.abs().nlargest(8).index]
s6_remainder = float(s6_contributions.sum() - s6_top.contribution.sum())
print("Validation ID:", s6_record.sample_id)
print(s6_top.sort_values("contribution").to_string(index=False))
print(f"Bias {s6_bias:.6f}; other contributions {s6_remainder:.6f}")
print(f"FULL sum z={s6_z:.6f}; sigmoid P(B)={s6_p:.6f}")
s6_view = s6_top.sort_values("contribution")
plt.figure(figsize=(8, 4))
plt.barh(s6_view.kmer, s6_view.contribution,
         color=["#D55E00" if value >= 0 else "#0072B2"
                for value in s6_view.contribution])
plt.axvline(0, color="#697782", linewidth=1)
plt.xlabel("Contribution to score (count × coefficient)")
plt.title("Selected contributions; full sum includes all features and bias")
plt.tight_layout()
plt.show()
plt.close("all")


In [ ]:
s6_coefficients = pd.read_csv(s3_destination / "kmer_coefficients.csv")
s6_importance = pd.read_csv(s3_destination / "random_forest_importance.csv")
s6_columns = ["kmer", "motif_window_probability", "motif_vs_background_enrichment"]
print("Signed global coefficients; not count-dependent local contributions:")
print(s6_coefficients[["coefficient"] + s6_columns].head(6).to_string(index=False))
print("Nonnegative global forest impurity importance; no positive/negative direction:")
print(s6_importance[["importance"] + s6_columns].head(6).to_string(index=False))
print("Enrichment = mean internal PPM-window probability / (1/256).")
print("It excludes boundary windows and class mixtures; it is not a causal claim.")
assert s6_importance.importance.ge(0).all()
np.testing.assert_allclose(s6_importance.importance.sum(), 1.0)
np.testing.assert_allclose(s6_coefficients.motif_window_probability.sum(), 1.0)
print("Discuss: which clue mattered here, and what would you still want to check?")


**End of Day 1.** You followed DNA → counts → fitting → probabilities →
evaluation → explanation. Coefficients, local contributions, forest importance
and distribution-derived enrichment have different meanings and scales.
None establishes a real regulatory mechanism. Related families or batch
effects require appropriate evaluation; the separate leakage notebook is an
extension. Tomorrow we keep sequence position and learn representations.
No homework is required. No protected test result was calculated here.
